In [3]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langchain_groq import ChatGroq
from langchain_core.tools import tool

In [4]:
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [15]:
tools = [add]
llm = ChatGroq(model="openai/gpt-oss-120b")
llm_with_tools = llm.bind_tools(tools)

In [23]:
# Agent node
def call_model(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state['messages'])
    return {
        "messages": [response]
        }

In [24]:
# 5. Routing function
def should_continue(state: MessagesState) -> str:
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END

In [25]:
# 6. Build graph
builder = StateGraph(MessagesState)

builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools=tools))


# 7. Add edges
builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        END: END
    }
)

# After tool execution, return to LLM
builder.add_edge("tools", "agent")

In [26]:
graph = builder.compile()

In [27]:
# 9. Run
result = graph.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What's 17 plus 25?"
        }
    ]
})


print(result["messages"][-1].content)

17 + 25 = 42.
